# 08282024

## 1. How to debug in a pipeline(wrong, look at 0901 notebook):
* Encounter an error and speculate the possible causes.
* Isolate the problematic code. In a pipeline, just copy all code before the error-triggering command, especially the [global].
* Put the code in Python, adjust its format and replace placeholders with real path or variables. 
* Add logging to print message or export files to track variable states and flow.
* Fix the error based on the feedback.

I will use an example to display and fix the error regarding with `--sample_specific` option in RNA_calling.ipynb:
* Using correct sos command to submit mmjobman for three pairs of samples, I received error: `ERROR: [STAR_align_1]: [STAR_align_1]: Failed to process input statement raw_reads, group_by = is_paired_end + 1, group_with = {"sample_id","read_length"}: Length of provided properties (3) does not match number of groups (1)` on stderr.autosave of mmcloud interface.
* This error is from: `input: raw_reads, group_by = is_paired_end + 1, group_with = {"sample_id","read_length"}` in [STAR_align_1]
* Since [STAR_align_1] is at the beginning of the pipeline and the pipeline is not that long, I just run code in an interactive session with toy data to degub and didn't consider about the input replacement.
* Make changes below:
*I added four `print` commands before thr `input` command.*
**print(f"Number of raw reads: {len(raw_reads)}") # to show the number of samples input
  print(f"Number of sample IDs: {len(sample_id)}")# to show if the samples are one pair
  print(f"Number of read lengths: {len(read_length)}") # to show the number of one of the variables based on which inputs are grouped
  print(f"is_paired_end: {is_paired_end}") # to show if sample type is correctly identified and passed to this step(I know the samples are paired-end sequencing)**
  input: raw_reads, group_by = is_paired_end + 1, group_with = {"sample_id","read_length"}
*I added two print in the code chunk defining `--sample_specific` option*
'## Align specific sample    
if not (sample_specific == ''):
    print("Align sample",sample_specific, 'only...')
    sample_inv = sample_inv[sample_inv['ID'] == sample_specific]  
  **print("Filtered sample inventory:")
    print(sample_inv)** ## to see if this option did specify a pair of samples
*The feedback on the interactive session is as follow:*
`jovyan@3f0e23aab927:/data/interactive_analysis/rl3328$ sos run /data/interactive_analysis/rl3328/final_RNA_calling.ipynb STAR_align     --cwd /data/interactive_analysis/rl3328/test_output/star_output /     --samples /data/interactive_analysis/rl3328/toy/toy_list     --data-dir /data/interactive_analysis/rl3328/toy /     --STAR-index  /data/resource/References/STAR_Index /     --gtf /data/resource/References/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.gtf     --reference-fasta /data/resource/References/GRCh38_full_analysis_set_plus_decoy_hla.noALT_noHLA_noDecoy_ERCC.fasta     --ref-flat /data/resource/References/Homo_sapiens.GRCh38.103.chr.reformatted.ERCC.ref.flat     --varVCFfile /data/resource/References/ZOD14598_AD_GRM_WGS_2021-04-29_all.recalibrated_variants.leftnorm.filtered.AF.WASP.vcf     --sample_specific sample_1_short
INFO: NumExpr defaulting to 4 threads.
Align sample sample_1_short only...
Filtered sample inventory:
               ID                     fq1                     fq2
0  sample_1_short  sample_1_short.1.fq.gz  sample_1_short.2.fq.gz
INFO: Input samples are paired-end sequences.
INFO: Running STAR_align_1: 
Insufficent memory for STAR, changing to 40G
Adding wasp filter in STAR alignment
Number of raw reads: 2
Number of sample IDs: 1
Number of read lengths: 3
is_paired_end: 1
ERROR: [STAR_align_1]: [STAR_align_1]: Failed to process input statement raw_reads, group_by = is_paired_end + 1, group_with = {"sample_id","read_length"}
: Length of provided properties (3) does not match number of groups (1)
[STAR_align]: Exits with 4 pending steps (STAR_align_2, STAR_align_3, strand_detected_1, strand_detected_2)`

* The feedback showed that Number of read lengths is 3, indicating the inputs were grouped by the whole list of read_length in the folder(I only include three pairs of samples in the `--data_dir` directory). But it should be the read_length of the current running samples.
*I first changed "read_length" to "_read_length" in `input: raw_reads, group_by = is_paired_end + 1, group_with = {"sample_id","read_length"}`, but not worked.*
*Then, I switch the reading order of these variables in [global], first # Align specific sample, then # exact read_length. It worked.*









## 2. Difference between read_length and _read_length in RNA_calling.ipynb
* read_length:
This is a parameter that is defined at the workflow level.
It represents the overall read length that is used for the entire set of samples.
If _read_length is 0 for a sample, then the workflow will use the value of read_length to set the --sjdbOverhang parameter in the STAR aligner command.
* _read_length:
This is a variable that is specific to the current sample being processed.
It represents the read length for the current sample.
This value is likely set or passed in as part of the input data for each sample.
It is used to determine the appropriate value for the --sjdbOverhang parameter in the STAR aligner command.

